# Predicting Low-Carbon Hours for Data-Center Computing

**Course:** AC209A  

**Canvas project group:** 14 

**Group members:** Minh-Thy Tyler, Gabe Gonzalez, Noah Schiro

## 1. Getting the Data

We obtained the PJM balancing authority Excel file from the EIA
Hourly Electric Grid Monitor.

Source: https://www.eia.gov/electricity/gridmonitor/knownissues/xls/PJM.xlsx

Download date: November 2 2026

The original file is stored locally at `data/raw/PJM.xlsx`.
We will inspect its sheets and date coverage before selecting our
proposed study period of January 2022 through December 2025.

In [4]:
# packages
from pathlib import Path
import pandas as pd

In [6]:
data_path = Path("../data/raw/PJM.xlsx")

#report download files size
size_mb = data_path.stat().st_size / 1_000_000
print(f"Downloaded data size: {size_mb:.2f} MB")

#inspecing available sheets before choosing which ones to load 
excel_file = pd.ExcelFile(data_path)
print("Available sheets:", excel_file.sheet_names)

Downloaded data size: 114.54 MB
Available sheets: ['Published Hourly Data', 'Published Daily Data', 'BA Full Names', 'Hourly Charts', 'Daily Charts', 'Known Data Issues', 'Notes', 'Imbalance Thresholds', 'Range Thresholds']


In [8]:
preview = pd.read_excel(
    excel_file,
    sheet_name="Published Hourly Data",
    nrows=5
)

display(preview)
print(preview.columns.tolist())

,BA,UTC time,Local date,Hour,Local time,Time zone,Generation only?,Demand forecast,Demand,Net generation,...,CO2 Emissions: OIL,CO2 Emissions: Other,CO2 Emissions Generated,CO2 Emissions Imported,CO2 Emissions Exported,CO2 Emissions Consumed,Positive Generation,Consumed Electricity,CO2 Emissions Intensity for Generated Electricity,CO2 Emissions Intensity for Consumed Electricity
0,PJM,2015-07-01 05:00:00,2015-07-01,1,2015-07-01 01:00:00,Eastern,N,29415,84024,83944,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,PJM,2015-07-01 06:00:00,2015-07-01,2,2015-07-01 02:00:00,Eastern,N,27687,79791,80091,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,PJM,2015-07-01 07:00:00,2015-07-01,3,2015-07-01 03:00:00,Eastern,N,26574,76760,77994,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,PJM,2015-07-01 08:00:00,2015-07-01,4,2015-07-01 04:00:00,Eastern,N,26029,74931,76820,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,PJM,2015-07-01 09:00:00,2015-07-01,5,2015-07-01 05:00:00,Eastern,N,26220,74368,76268,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


['BA', 'UTC time', 'Local date', 'Hour', 'Local time', 'Time zone', 'Generation only?', 'Demand forecast', 'Demand', 'Net generation', 'Total interchange', 'Imputed demand', 'Imputed net generation', 'Imputed total interchange', 'Adjusted demand', 'Adjusted net generation', 'Adjusted total interchange', 'NG: COL', 'NG: NG', 'NG: NUC', 'NG: OIL', 'NG: GEO', 'NG: WAT', 'NG: PS', 'NG: SUN', 'NG: SNB', 'NG: WND', 'NG: WNB', 'NG: BAT', 'NG: OES', 'NG: UES', 'NG: OTH', 'NG: UNK', 'Imputed COL Gen', 'Imputed NG Gen', 'Imputed NUC Gen', 'Imputed OIL Gen', 'Imputed GEO Gen', 'Imputed WAT Gen', 'Imputed PS Gen', 'Imputed SUN Gen', 'Imputed SNB Gen', 'Imputed WND Gen', 'Imputed WNB Gen', 'Imputed BAT Gen', 'Imputed OES Gen', 'Imputed UES Gen', 'Imputed OTH Gen', 'Imputed UNK Gen', 'Adjusted COL Gen', 'Adjusted NG Gen', 'Adjusted NUC Gen', 'Adjusted OIL Gen', 'Adjusted GEO Gen', 'Adjusted WAT Gen', 'Adjusted PS Gen', 'Adjusted SUN Gen', 'Adjusted SNB Gen', 'Adjusted WND Gen', 'Adjusted WNB Gen', '

In [9]:
hourly = pd.read_excel(
    excel_file,
    sheet_name="Published Hourly Data"
)

hourly["UTC time"] = pd.to_datetime(hourly["UTC time"])

print("Rows and columns:", hourly.shape)
print("Earliest timestamp:", hourly["UTC time"].min())
print("Latest timestamp:", hourly["UTC time"].max())

Rows and columns: (99409, 108)
Earliest timestamp: 2015-07-01 05:00:00
Latest timestamp: 2026-11-02 05:00:00


### Creating separate subset for our study

In [10]:
local_dates = pd.to_datetime(hourly["Local date"])

pjm = hourly.loc[
    (local_dates >= "2022-01-01") &
    (local_dates < "2026-01-01")
].copy()

print("Study-period rows and columns:", pjm.shape)
print("First local date:", pjm["Local date"].min())
print("Last local date:", pjm["Local date"].max())

Study-period rows and columns: (35064, 108)
First local date: 2022-01-01 00:00:00
Last local date: 2025-12-31 00:00:00


In [11]:
for column in pjm.columns:
    print(column)

BA
UTC time
Local date
Hour
Local time
Time zone
Generation only?
Demand forecast
Demand
Net generation
Total interchange
Imputed demand
Imputed net generation
Imputed total interchange
Adjusted demand
Adjusted net generation
Adjusted total interchange
NG: COL
NG: NG
NG: NUC
NG: OIL
NG: GEO
NG: WAT
NG: PS
NG: SUN
NG: SNB
NG: WND
NG: WNB
NG: BAT
NG: OES
NG: UES
NG: OTH
NG: UNK
Imputed COL Gen
Imputed NG Gen
Imputed NUC Gen
Imputed OIL Gen
Imputed GEO Gen
Imputed WAT Gen
Imputed PS Gen
Imputed SUN Gen
Imputed SNB Gen
Imputed WND Gen
Imputed WNB Gen
Imputed BAT Gen
Imputed OES Gen
Imputed UES Gen
Imputed OTH Gen
Imputed UNK Gen
Adjusted COL Gen
Adjusted NG Gen
Adjusted NUC Gen
Adjusted OIL Gen
Adjusted GEO Gen
Adjusted WAT Gen
Adjusted PS Gen
Adjusted SUN Gen
Adjusted SNB Gen
Adjusted WND Gen
Adjusted WNB Gen
Adjusted BAT Gen
Adjusted OES Gen
Adjusted UES Gen
Adjusted OTH Gen
Adjusted UNK Gen
CPLE
CPLW
DUK
LGEE
MISO
NYIS
OVEC
TVA
Subregion AP
Subregion AEP
Subregion ATSI
Subregion AE
Subr